In [1]:
# Correctly import the ProjectManager class
from src.project_manager import ProjectManager
import sys
import os

# Ensure the src and ux_ui directories are in the Python path
sys.path.append(os.path.abspath('./src'))
sys.path.append(os.path.abspath('./ux_ui'))

# Initialize the ProjectManager with the base directory
base_directory = '../data/v3.0_las_files'
project_manager_instance = ProjectManager(base_directory)
project_manager_instance.selected_field = 'Scoda'

# Load the project with progress callback (set to None since we're not using UI)
project = project_manager_instance.load_selected_field(progress_callback=None)

project_manager_instance.selected_curves = ['DCAL', 'SCAL', 'MCAL', 'GR', 'SP', 'MN', 'MI',
                                            'RILM', 'RILD', 'RLL3', 'RXORT', 'RHOB', 'RHOC', 'CILD', 'DPOR', 'SPOR', 'DT', 'CNLS']

project_manager_instance.standardized_curve_mapping = {
    'Cali': ['DCAL', 'SCAL', 'MCAL'],
    'GR-SP': ['GR', 'SP'],
    'Micro': ['MN', 'MI'],
    'RIL': ['RILM', 'RILD', 'RLL3', 'RXORT'],
    'Density': ['RHOB', 'RHOC', 'CILD', 'DPOR'],
    'Sonic': ['SPOR', 'DT'],
    'Neutron': ['CNLS']
}

# Run funtion to determined outliers
project_manager_instance.detect_all_outliers()

# Filter outliers (make sure to call the method with parentheses)
project_manager_instance.prepare_data()

data = project_manager_instance.prepared_data
selected_curves = ['Cali', 'GR', 'SP', 'MN', 'MI', 'RILM', 'RILD',
                   'RLL3', 'RXORT', 'RHOB', 'RHOC', 'CILD', 'DPOR', 'SPOR', 'DT']
curves_to_predict = ['CNLS', 'Formation']
unique_formations = project_manager_instance.unique_formations


Data has been prepared successfully.


In [2]:
from src.neural_network.pipeline import pipeline
from src.neural_network.hyperparameters import *

# Limpiar VRAM antes de ejecutar
import gc
import tensorflow as tf
from src.utils.memory_manager import clean_memory_for_trial

# Limpiar completamente
tf.keras.backend.clear_session()
clean_memory_for_trial()
gc.collect()

(
    data_corrected, correction_report,                                          # Step 0.5
    train_validation_data, external_test_data, discarded_wells,                 # Step 1
    engineered_data, feature_info, final_cols,                                  # Step 2 
    X_scaled, y_scaled, per_well_strategies, global_feature_scalers,            # Step 3
        categorical_encoders, column_types, feature_columns, global_columns, 
        well_descriptors, target_scalers, formation_encoder, unknown_index, 
        normalizers, fit_errors,
    top_configs, study,                                                         # Step 4
    # best_config, cv_results, best_model,                                        # Step 5
    # model, history,                                                             # Step 6
    # predictions,                                                                # Step 7
) = pipeline(
    data, selected_curves, curves_to_predict
)


    

ModuleNotFoundError: No module named 'src.utils'

In [ ]:
# ----
# Step 1 – Analyze Formation Class Distributions with Formation Mapping
# ----

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import sys
import os

# Add formation_mapper to path
sys.path.append('/workspace/code/src/neural_network')
from formation_mapper import get_formation_mapping, standardize_formation_name

print("🔍 Formation Class Distribution Analysis with Mapping")
print("=" * 60)

# Substep 1.1 – Get formation mapping ______________________
formation_mapping = get_formation_mapping()
print(f"📋 Formation mapping loaded: {len(formation_mapping)} mappings")

# Substep 1.2 – Extract and map formation data ______________________
all_formations_original = []
all_formations_mapped = []
well_formation_counts_original = {}
well_formation_counts_mapped = {}

for well_name, well_data in train_validation_data.items():
    if 'Formation' in well_data.columns:
        formations_original = well_data['Formation'].dropna()
        formations_mapped = formations_original.apply(standardize_formation_name)
        
        all_formations_original.extend(formations_original.tolist())
        all_formations_mapped.extend(formations_mapped.tolist())
        
        # Count formations per well (original and mapped)
        well_formation_counts_original[well_name] = formations_original.value_counts().to_dict()
        well_formation_counts_mapped[well_name] = formations_mapped.value_counts().to_dict()
        
        print(f"📊 {well_name}: {len(formations_original)} formation points")

# Substep 1.3 – Calculate distributions ______________________
original_distribution = Counter(all_formations_original)
mapped_distribution = Counter(all_formations_mapped)
total_points = len(all_formations_original)

print(f"\n📈 ORIGINAL Formation Distribution ({total_points} total points):")
print("-" * 60)
for formation, count in original_distribution.most_common():
    percentage = (count / total_points) * 100
    print(f"   {formation:<30}: {count:>6} points ({percentage:>5.1f}%)")

print(f"\n📈 MAPPED Formation Distribution ({total_points} total points):")
print("-" * 60)
for formation, count in mapped_distribution.most_common():
    percentage = (count / total_points) * 100
    print(f"   {formation:<30}: {count:>6} points ({percentage:>5.1f}%)")

# Substep 1.4 – Show mapping effects ______________________
print(f"\n🔄 Mapping Effects:")
print("-" * 30)
print(f"   📊 Original unique formations: {len(original_distribution)}")
print(f"   📊 Mapped unique formations: {len(mapped_distribution)}")
print(f"   📉 Reduction: {len(original_distribution) - len(mapped_distribution)} formations")

# Show which formations were consolidated
consolidated_groups = {}
for original, mapped in formation_mapping.items():
    if mapped not in consolidated_groups:
        consolidated_groups[mapped] = []
    if original != mapped:  # Only show actual mappings
        consolidated_groups[mapped].append(original)

print(f"\n🔗 Formation Consolidations:")
for mapped_name, original_names in consolidated_groups.items():
    if original_names:  # Only show groups that had consolidations
        original_counts = [original_distribution.get(name, 0) for name in original_names]
        total_consolidated = sum(original_counts)
        if total_consolidated > 0:
            print(f"   {mapped_name}:")
            for orig_name in original_names:
                if original_distribution.get(orig_name, 0) > 0:
                    print(f"      • {orig_name}: {original_distribution[orig_name]} points")
            print(f"      → Total consolidated: {total_consolidated} points")

# Substep 1.5 – Create comparative visualization ______________________
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 15))

# Plot 1: Original distribution
formations_orig = [item[0] for item in original_distribution.most_common()]
counts_orig = [item[1] for item in original_distribution.most_common()]

bars1 = ax1.bar(range(len(formations_orig)), counts_orig, color='lightcoral', alpha=0.7)
ax1.set_xlabel('Formation')
ax1.set_ylabel('Number of Points')
ax1.set_title('Original Formation Distribution')
ax1.set_xticks(range(len(formations_orig)))
ax1.set_xticklabels(formations_orig, rotation=45, ha='right')

# Add count labels
for bar, count in zip(bars1, counts_orig):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts_orig)*0.01, 
             str(count), ha='center', va='bottom', fontsize=8)

# Plot 2: Mapped distribution
formations_mapped = [item[0] for item in mapped_distribution.most_common()]
counts_mapped = [item[1] for item in mapped_distribution.most_common()]

bars2 = ax2.bar(range(len(formations_mapped)), counts_mapped, color='skyblue', alpha=0.7)
ax2.set_xlabel('Formation')
ax2.set_ylabel('Number of Points')
ax2.set_title('Mapped Formation Distribution')
ax2.set_xticks(range(len(formations_mapped)))
ax2.set_xticklabels(formations_mapped, rotation=45, ha='right')

# Add count labels
for bar, count in zip(bars2, counts_mapped):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts_mapped)*0.01, 
             str(count), ha='center', va='bottom', fontsize=8)

# Plot 3: Original per-well heatmap
well_names = list(well_formation_counts_original.keys())
orig_formation_names = list(original_distribution.keys())

heatmap_data_orig = []
for well in well_names:
    well_row = []
    for formation in orig_formation_names:
        count = well_formation_counts_original[well].get(formation, 0)
        well_row.append(count)
    heatmap_data_orig.append(well_row)

heatmap_df_orig = pd.DataFrame(heatmap_data_orig, index=well_names, columns=orig_formation_names)
sns.heatmap(heatmap_df_orig, annot=True, fmt='d', cmap='Reds', ax=ax3, 
            cbar_kws={'label': 'Point Count'})
ax3.set_title('Original Formation Distribution by Well')
ax3.set_xlabel('Formation')
ax3.set_ylabel('Well Name')

# Plot 4: Mapped per-well heatmap
mapped_formation_names = list(mapped_distribution.keys())

heatmap_data_mapped = []
for well in well_names:
    well_row = []
    for formation in mapped_formation_names:
        count = well_formation_counts_mapped[well].get(formation, 0)
        well_row.append(count)
    heatmap_data_mapped.append(well_row)

heatmap_df_mapped = pd.DataFrame(heatmap_data_mapped, index=well_names, columns=mapped_formation_names)
sns.heatmap(heatmap_df_mapped, annot=True, fmt='d', cmap='Blues', ax=ax4, 
            cbar_kws={'label': 'Point Count'})
ax4.set_title('Mapped Formation Distribution by Well')
ax4.set_xlabel('Formation')
ax4.set_ylabel('Well Name')

plt.tight_layout()
plt.show()

# Substep 1.6 – Class imbalance analysis (both original and mapped) ______________________
print(f"\n⚖️ Class Imbalance Analysis:")
print("=" * 40)

# Original imbalance
max_count_orig = max(original_distribution.values())
min_count_orig = min(original_distribution.values())
imbalance_ratio_orig = max_count_orig / min_count_orig

print(f"📊 ORIGINAL DATA:")
print(f"   Most common: {max(original_distribution, key=original_distribution.get)} ({max_count_orig} points)")
print(f"   Least common: {min(original_distribution, key=original_distribution.get)} ({min_count_orig} points)")
print(f"   Imbalance ratio: {imbalance_ratio_orig:.1f}:1")

# Mapped imbalance
max_count_mapped = max(mapped_distribution.values())
min_count_mapped = min(mapped_distribution.values())
imbalance_ratio_mapped = max_count_mapped / min_count_mapped

print(f"\n📊 MAPPED DATA:")
print(f"   Most common: {max(mapped_distribution, key=mapped_distribution.get)} ({max_count_mapped} points)")
print(f"   Least common: {min(mapped_distribution, key=mapped_distribution.get)} ({min_count_mapped} points)")
print(f"   Imbalance ratio: {imbalance_ratio_mapped:.1f}:1")

print(f"\n📈 IMPROVEMENT:")
print(f"   Imbalance reduction: {imbalance_ratio_orig:.1f}:1 → {imbalance_ratio_mapped:.1f}:1")
print(f"   Improvement factor: {imbalance_ratio_orig/imbalance_ratio_mapped:.1f}x better")

# Identify severely underrepresented classes
print(f"\n⚠️ Severely underrepresented formations (< 1%):")
print("   ORIGINAL:")
for formation, count in original_distribution.items():
    percentage = (count / total_points) * 100
    if percentage < 1.0:
        print(f"      • {formation}: {count} points ({percentage:.2f}%)")

print("   MAPPED:")
for formation, count in mapped_distribution.items():
    percentage = (count / total_points) * 100
    if percentage < 1.0:
        print(f"      • {formation}: {count} points ({percentage:.2f}%)")



In [ ]:
import os
import sys
%matplotlib inline

# Cambiar al directorio workspace
os.chdir('/workspace')
print(f"📁 Directorio actual: {os.getcwd()}")

# Agregar utils al path
utils_path = os.path.join(os.getcwd(), 'code', 'src', 'utils')
sys.path.insert(0, utils_path)

print(f"📂 Utils path: {utils_path}")
print(f"✅ Existe: {os.path.exists(utils_path)}")

# Importar
from plot_logs import plot_predicted_curves
plot_predicted_curves(save_plot=True, show_plot=True)